In [7]:
## STEP 1 IMPORT LIBRARY

import os
import pandas as pd
import numpy as np
import librosa
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (confusion_matrix, accuracy_score,
                             precision_score, recall_score, f1_score,
                             classification_report)
import warnings
warnings.filterwarnings('ignore')

In [8]:
audio_folder = "clips"

valid_files = []
corrupt_files = []

for filename in os.listdir(audio_folder):
    file_path = os.path.join(audio_folder, filename)

    # Lewati folder/subfolder
    if not os.path.isfile(file_path):
        continue

    try:
        librosa.load(file_path, sr=None)
        valid_files.append(filename)
    except Exception as e:
        corrupt_files.append({
            "file": filename,
            "error": str(e)
        })

print("Total file valid :", len(valid_files))
print("Total file rusak :", len(corrupt_files))

if corrupt_files:
    print("\nDaftar file rusak:")
    for item in corrupt_files:
        print(item["file"], "->", item["error"])
else:
    print("\nTidak ada file audio rusak.")

Total file valid : 60451
Total file rusak : 0

Tidak ada file audio rusak.


In [9]:
## STEP 2 LOAD DATASET

# Path ke file metadata dan folder audio
metadata_path = 'validated.tsv'
audio_folder = 'clips'

# Membaca metadata
df = pd.read_csv(metadata_path, sep='\t')
print(f"Jumlah data awal: {len(df)}")
print(df.head())

Jumlah data awal: 30331
                                           client_id  \
0  00f046cd1abd7e4ed354c2d97b67aceee8deb20bf53e5c...   
1  018b3f1a0a941a5b881a886830d9f133402ed66409489a...   
2  025f07759891360d312521dd83679fa6a233b9d7aa9d4a...   
3  0a4e7e60acc2a6145fc2589e6abb81e41e430cea0611bd...   
4  0a64c046530244b7507d4404575e8f673613c939ad6fac...   

                           path  \
0  common_voice_id_39599471.mp3   
1  common_voice_id_32165882.mp3   
2  common_voice_id_27657418.mp3   
3  common_voice_id_23033570.mp3   
4  common_voice_id_40088845.mp3   

                                         sentence_id  \
0  05e82b3c6f238b34985f158c4b63a8a9fda0382f1bd798...   
1  dd457d82f974202742803bf426f129d339fb1cf2217876...   
2  34f56b62e91fbbffeb7df499d4486aaa9f6d963a93d57f...   
3  ecee266f978e8296fc17d2bcfa744429165db882dbe5fd...   
4  67125efffe8d90c1147d6d609ca9e041f17b1a78b6a3d2...   

                                            sentence  sentence_domain  \
0                D

In [10]:
y, sr = librosa.load(...)

y = librosa.util.normalize(y)

TypeError: Invalid file: Ellipsis

In [ ]:
## STEP 3 FILTER DATASET

# Filter hanya dua kelas yang diinginkan
df_filtered = df[df['gender'].isin(['male_masculine', 'female_feminine'])].copy()
print(f"Jumlah data setelah filter: {len(df_filtered)}")
print("Distribusi gender:")
print(df_filtered['gender'].value_counts())

Jumlah data setelah filter: 21587
Distribusi gender:
gender
male_masculine     13983
female_feminine     7604
Name: count, dtype: int64


In [ ]:
## STEP 4 VALIDATE AUDIO FILES

# Daftar path file audio yang valid
valid_paths = []
for idx, row in df_filtered.iterrows():
    audio_path = os.path.join(audio_folder, row['path'])
    if os.path.exists(audio_path):
        valid_paths.append(idx)

df_filtered = df_filtered.loc[valid_paths].reset_index(drop=True)
print(f"Jumlah data setelah validasi file: {len(df_filtered)}")

Jumlah data setelah validasi file: 21587


In [ ]:
##STEP 5 Menentukan parameter eksperimen

# Parameter eksperimen
N_MFCC = 13          # M1
K = 3                # K1

print(f"Eksperimen: MFCC={N_MFCC}, K={K}")

Eksperimen: MFCC=13, K=3


In [ ]:
pre = 0.97

y = np.append(
    y[0],
    y[1:] - pre * y[:-1]
)

In [ ]:
##STEP 6 Ekstrak Fitur MFCC

def extract_mfcc(file_path, n_mfcc=13):
    try:
        # Muat audio dengan sampling rate default 22050 Hz (Librosa default)
        y, sr = librosa.load(file_path, sr=None)  # sr=None biar pakai asli
        # Ekstrak MFCC
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        # Ambil rata-rata tiap koefisien sepanjang waktu
        mfcc_mean = np.mean(mfcc, axis=1)
        return mfcc_mean
    except Exception as e:
        print(f"Error pada file {file_path}: {e}")
        return None

In [ ]:
##STEP 7 Ekstrak fitur untuk semua file audio

X = []
y = []

for idx, row in df_filtered.iterrows():
    audio_path = os.path.join(audio_folder, row['path'])
    features = extract_mfcc(audio_path, n_mfcc=N_MFCC)
    if features is not None:
        X.append(features)
        y.append(row['gender'])

# Konversi ke numpy array
X = np.array(X)
y = np.array(y)

print(f"Jumlah fitur yang berhasil diekstrak: {X.shape[0]}")
print(f"Dimensi fitur: {X.shape[1]}")  # Harus = N_MFCC

Jumlah fitur yang berhasil diekstrak: 21587
Dimensi fitur: 13


In [ ]:
##STEP 8 Distribusi label setelah ekstraksi

unique, counts = np.unique(y, return_counts=True)
print("Distribusi label setelah ekstraksi:")
for label, count in zip(unique, counts):
    print(f"{label}: {count}")

Distribusi label setelah ekstraksi:
female_feminine: 7604
male_masculine: 13983


In [ ]:
## STEP 9 Encode label

le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

Label mapping: {np.str_('female_feminine'): np.int64(0), np.str_('male_masculine'): np.int64(1)}


In [ ]:
##STEP 10 Split data menjadi training dan testing

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    stratify=y_encoded
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

Training set size: 17269
Testing set size: 4318


In [ ]:
## STEP 11 Standarisasi fitur

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Standarisasi selesai.")

Standarisasi selesai.


In [ ]:
## STEP 12 Buat model KNN

knn = KNeighborsClassifier(n_neighbors=K)
print(f"Model KNN dengan K={K} dibuat.")

Model KNN dengan K=3 dibuat.


In [ ]:
##STEP 13 Latih model KNN

knn.fit(X_train_scaled, y_train)
print("Training selesai.")

Training selesai.


In [ ]:
##STEP 14 Prediksi pada data testing

y_pred = knn.predict(X_test_scaled)
print("Prediksi selesai.")

Prediksi selesai.


In [ ]:
## STEP 15 Evaluasi model

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# Metrik
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='binary')  # karena binary
rec = recall_score(y_test, y_pred, average='binary')
f1 = f1_score(y_test, y_pred, average='binary')

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-Score  : {f1:.4f}")

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Confusion Matrix:
[[1462   59]
 [  53 2744]]
Accuracy  : 0.9741
Precision : 0.9790
Recall    : 0.9811
F1-Score  : 0.9800

Classification Report:
                 precision    recall  f1-score   support

female_feminine       0.97      0.96      0.96      1521
 male_masculine       0.98      0.98      0.98      2797

       accuracy                           0.97      4318
      macro avg       0.97      0.97      0.97      4318
   weighted avg       0.97      0.97      0.97      4318



In [ ]:
##STEP 16 Tampilkan hasil eksperimen

print("\n=== HASIL EKSPERIMEN ===")
print(f"MFCC = {N_MFCC}, K = {K}")
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-Score  : {f1:.4f}")
print("Confusion Matrix:")
print(cm)


=== HASIL EKSPERIMEN ===
MFCC = 13, K = 3
Accuracy  : 0.9741
Precision : 0.9790
Recall    : 0.9811
F1-Score  : 0.9800
Confusion Matrix:
[[1462   59]
 [  53 2744]]
